In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df_food.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# Missing values

# Drop rows where target (Delivery_Time) or key features are missing - can't predict without them
print(f"Before: {df_food.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Courier_Experience_yrs'])

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Time_of_Day', 'Traffic_Level']:
    df_clean[col] = df_clean[col].fillna('unknown')
print(f"After: {df_clean.shape}")
print("Missing values remaining:", df_clean.isnull().sum().sum()) #  sums column‑wise, the second sum() adds up the values in that Series.

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

In [ ]:
# Task 5: Write your code here:
numerical_cols =  df_clean.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Vehicle_Type',
                'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse

RF_mse = []

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1): # enumerate adds a counter to the loop.

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train
    model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Validate
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    mse = sklearn_mse(y_test, y_pred)

    # Store results
    RF_mse.append(mse)

print(f"  Average MSE: {np.mean(RF_mse):.4f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: